# PaySim — Data Cleaning & Undersampling

## Cell 1 — Imports

In [11]:
import pandas as pd
import numpy as np
from imblearn.under_sampling import RandomUnderSampler


## Cell 2 — Load Data

In [12]:
df = pd.read_csv('data/data.csv')

print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nClass distribution:')
print(df['isFraud'].value_counts())
print('\nNull values:')
print(df.isnull().sum())

Shape: (6362620, 11)

Columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']

Class distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64

Null values:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64


## Cell 3 — Drop Useless Columns

## Cell 4 — Drop Null Values

In [13]:
before = len(df)
df.dropna(inplace=True)
after = len(df)


## Cell 5 — Drop Duplicates

In [14]:
before = len(df)
df.drop_duplicates(subset=['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig'], inplace=True)
after = len(df)

print(f'Removed {before - after} duplicate rows')
print(f'Shape after dedup: {df.shape}')

Removed 1746 duplicate rows
Shape after dedup: (6360874, 11)


## Cell 6 — Undersample Majority (delete from 0s)

In [15]:
X = df.drop(columns=['isFraud'])
y = df['isFraud']

fraud_count = y.sum()
majority_target = int(fraud_count * (60 / 40))  

print(f'Fraud samples (1s)   : {fraud_count}')
print(f'Keeping majority (0s): {majority_target}')

under = RandomUnderSampler(
    sampling_strategy={0: majority_target},
    random_state=42
)
X_under, y_under = under.fit_resample(X, y)

print('\nAfter undersampling:')
print(pd.Series(y_under).value_counts())


Fraud samples (1s)   : 8158
Keeping majority (0s): 12237

After undersampling:
isFraud
0    12237
1     8158
Name: count, dtype: int64


## Cell 7 — Final Check

In [16]:
final_df = pd.DataFrame(X_under, columns=X.columns)
final_df['isFraud'] = y_under

print('Final shape:', final_df.shape)
print('\nClass distribution:')
print(final_df['isFraud'].value_counts())


Final shape: (20395, 11)

Class distribution:
isFraud
0    12237
1     8158
Name: count, dtype: int64


## Cell 8 — Save Cleaned & Undersampled Data

In [17]:
final_df.to_csv('data/modeldata.csv', index=False)

print('Saved to data/modeldata.csv')
print('Shape:', final_df.shape)

Saved to data/modeldata.csv
Shape: (20395, 11)
